# 🏦 Credit Risk Analyzer — Preprocessing Pipeline
## Home Credit Default Risk Dataset
- **Goal:** Clean the data, fix anomalies, engineer features, and prepare for ML modeling
- **Input:** application_train.csv (307,511 rows × 122 columns)
- **Output:** processed_data.csv — ready for XGBoost training

In [2]:
# Data manipulation
import pandas as pd
import numpy as np

# Saving processed data
import os

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [3]:
# Load raw data
df = pd.read_csv('../data/raw/application_train.csv')
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Shape: (307511, 122)
Rows: 307,511
Columns: 122


In [4]:
# Step 1 — Fix DAYS_EMPLOYED anomaly
# 365243 = placeholder for unemployed/retired applicants (~1000 years)
# Replace with NaN — permanent fix in preprocessing pipeline

before = (df['DAYS_EMPLOYED'] == 365243).sum()
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
after = df['DAYS_EMPLOYED'].isnull().sum() - (df['DAYS_EMPLOYED'].isnull().sum() - before)

print(f"365243 values replaced: {before:,}")
print(f"DAYS_EMPLOYED null count now: {df['DAYS_EMPLOYED'].isnull().sum():,}")
print("✅ DAYS_EMPLOYED anomaly fixed!")

365243 values replaced: 55,374
DAYS_EMPLOYED null count now: 55,374
✅ DAYS_EMPLOYED anomaly fixed!


In [5]:
# Step 2 — Feature Engineering
# DAYS_BIRTH and DAYS_EMPLOYED are stored as negative days — convert to years

df['AGE_YEARS'] = (df['DAYS_BIRTH'] / -365).astype(int)
df['EMPLOYED_YEARS'] = (df['DAYS_EMPLOYED'] / -365).round(1)
# DAYS_EMPLOYED has NaN values (365243 replaced) — NaN will be preserved here

print("New features created:")
print(f"AGE_YEARS — min: {df['AGE_YEARS'].min()}, max: {df['AGE_YEARS'].max()}")
print(f"EMPLOYED_YEARS — min: {df['EMPLOYED_YEARS'].min():.1f}, max: {df['EMPLOYED_YEARS'].max():.1f}")
print(f"EMPLOYED_YEARS null count: {df['EMPLOYED_YEARS'].isnull().sum():,}")
print("✅ Feature engineering done!")

# Defragment the DataFrame for better performance
df = df.copy()

New features created:
AGE_YEARS — min: 20, max: 69
EMPLOYED_YEARS — min: 0.0, max: 49.1
EMPLOYED_YEARS null count: 55,374
✅ Feature engineering done!


C:\Users\SHRIYUT\AppData\Local\Temp\ipykernel_14584\1295921859.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['AGE_YEARS'] = (df['DAYS_BIRTH'] / -365).astype(int)
C:\Users\SHRIYUT\AppData\Local\Temp\ipykernel_14584\1295921859.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['EMPLOYED_YEARS'] = (df['DAYS_EMPLOYED'] / -365).round(1)


In [6]:
# Step 3 — Create EMI ratio feature
# emi_ratio = monthly EMI (AMT_ANNUITY) / monthly income (AMT_INCOME_TOTAL)
# This directly maps to our rule engine logic

df['EMI_RATIO'] = (df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']).round(4)

print("EMI_RATIO feature created:")
print(f"Min: {df['EMI_RATIO'].min():.4f}")
print(f"Max: {df['EMI_RATIO'].max():.4f}")
print(f"Mean: {df['EMI_RATIO'].mean():.4f}")
print(f"Null count: {df['EMI_RATIO'].isnull().sum():,}")
print("✅ EMI_RATIO feature created!")

EMI_RATIO feature created:
Min: 0.0002
Max: 1.8760
Mean: 0.1809
Null count: 12
✅ EMI_RATIO feature created!


In [7]:
# Step 4 — Outlier capping using 99th percentile (Winsorization)
# Extreme outliers can distort model training — cap them at 99th percentile

cols_to_cap = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']

for col in cols_to_cap:
    cap_value = df[col].quantile(0.99)
    before_max = df[col].max()
    df[col] = df[col].clip(upper=cap_value)
    after_max = df[col].max()
    print(f"{col}:")
    print(f"  Before max: {before_max:,.0f} → After max: {after_max:,.0f}")

print("\n✅ Outlier capping done!")

AMT_INCOME_TOTAL:
  Before max: 117,000,000 → After max: 472,500
AMT_CREDIT:
  Before max: 4,050,000 → After max: 1,854,000
AMT_ANNUITY:
  Before max: 258,026 → After max: 70,006

✅ Outlier capping done!


In [8]:
# Step 5 — Missing value imputation
# Numerical columns: median imputation (robust to outliers)
# Categorical columns: mode imputation (most frequent value)

# Separate numerical and categorical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['str']).columns.tolist()
print(f"Numerical columns: {len(numerical_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")

# Median imputation for numerical columns
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

# Mode imputation for categorical columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)

# Verify
total_missing = df.isnull().sum().sum()
print(f"\nTotal missing values after imputation: {total_missing}")
print("✅ Missing value imputation done!")

df = df.copy()

Numerical columns: 109
Categorical columns: 16

Total missing values after imputation: 0
✅ Missing value imputation done!


In [9]:
# Step 6 — Categorical encoding
# Label encoding for binary columns (2 unique values)
# One-hot encoding for multi-category columns

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Identify binary vs multi-category columns
binary_cols = [col for col in categorical_cols if df[col].nunique() == 2]
multi_cols = [col for col in categorical_cols if df[col].nunique() > 2]

print(f"Binary columns (Label Encoding): {len(binary_cols)}")
print(f"Multi-category columns (One-Hot Encoding): {len(multi_cols)}")

# Label encode binary columns
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# One-hot encode multi-category columns
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print(f"\nShape after encoding: {df.shape}")
print("✅ Categorical encoding done!")

Binary columns (Label Encoding): 4
Multi-category columns (One-Hot Encoding): 12

Shape after encoding: (307511, 233)
✅ Categorical encoding done!


In [10]:
# Step 7 — Train/Test split
# 80% training, 20% testing
# stratify=True ensures both splits have same class distribution (92:8 ratio)

from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop('TARGET', axis=1)
y = df['TARGET']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\nTrain target distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))
print(f"\nTest target distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))


Features shape: (307511, 232)
Target shape: (307511,)

X_train shape: (246008, 232)
X_test shape: (61503, 232)

Train target distribution:
TARGET
0   91.93
1    8.07
Name: proportion, dtype: float64

Test target distribution:
TARGET
0   91.93
1    8.07
Name: proportion, dtype: float64


In [11]:
# Step 8 — Save processed data
# Save train and test sets separately for ML notebook

import os

output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

# Combine features and target for saving
train_df = X_train.copy()
train_df['TARGET'] = y_train.values

test_df = X_test.copy()
test_df['TARGET'] = y_test.values

# Save
train_df.to_csv(f'{output_dir}/train_processed.csv', index=False)
test_df.to_csv(f'{output_dir}/test_processed.csv', index=False)

print(f"Train set saved: {train_df.shape}")
print(f"Test set saved: {test_df.shape}")
print(f"Location: {output_dir}")
print("✅ Processed data saved!")

Train set saved: (246008, 233)
Test set saved: (61503, 233)
Location: ../data/processed
✅ Processed data saved!


## ✅ Preprocessing Summary

| Step | Task | Detail |
|---|---|---|
| 1 | DAYS_EMPLOYED anomaly fix | 55,374 values → NaN |
| 2 | Feature Engineering | AGE_YEARS, EMPLOYED_YEARS, EMI_RATIO created |
| 3 | Outlier Capping | 99th percentile winsorization on AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY |
| 4 | Missing Value Imputation | Median for numerical, Mode for categorical — 0 missing values remaining |
| 5 | Categorical Encoding | Label + One-Hot — 122 columns → 233 columns |
| 6 | Train/Test Split | 80:20 stratified — Train: 246,008 rows, Test: 61,503 rows |
| 7 | Data Saved | data/processed/train_processed.csv, test_processed.csv |

**Next Step → ML Model Training (Logistic Regression → Random Forest → XGBoost)**